<a href="https://colab.research.google.com/github/corebankingfiap-dev/core_banking/blob/main/01_Data_Lake_Commodities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Commodities

Bronze - Dado Bruto

Silver - Dado Limpo e Padronizado

Gold - Tranformado e Calculado

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. CONEXÃO E CONFIGURAÇÃO
if not os.path.exists('/content/drive'): drive.mount('/content/drive')

segmento = "commodities"
ativos = ['VALE3.SA', 'PETR4.SA', 'CSNA3.SA', 'SUZB3.SA', 'PRIO3.SA']
BASE_PATH = '/content/drive/MyDrive/DATA_LAKE/'

# Garantir pastas
for p in ['01_Bronze', '02_Silver', '03_Gold']:
    os.makedirs(os.path.join(BASE_PATH, p), exist_ok=True)

print(f"🚀 Iniciando Pipeline: {segmento.upper()}")

# --- ETAPA 1: BRONZE (O BRUTO) ---
df_bruto = yf.download(ativos, period="5y", interval="1d", auto_adjust=True)['Close']
df_bruto.to_csv(f'{BASE_PATH}01_Bronze/b3_{segmento}_bruto.csv')

# --- ETAPA 2: SILVER (O FORMATADO E O TÉCNICO) ---
df_limpo = df_bruto.ffill().sort_index()
df_limpo.columns = [col.replace('.SA', '') for col in df_limpo.columns]
df_limpo.to_csv(f'{BASE_PATH}02_Silver/b3_{segmento}_limpo.csv') # Técnico para cálculos

# Versão Visual com R$
def fmt_money(x): return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
df_formatado = df_limpo.map(fmt_money)
df_formatado.to_csv(f'{BASE_PATH}02_Silver/b3_{segmento}_formatados.csv')

# --- ETAPA 3: GOLD (A ANÁLISE) ---
log_retornos = np.log(df_limpo / df_limpo.shift(1)).dropna()
ret_anual = log_retornos.mean() * 252
vol_anual = log_retornos.std() * np.sqrt(252)
sharpe = ret_anual / vol_anual

df_gold = pd.DataFrame({
    'Retorno Anualizado (%)': (ret_anual * 100).round(2),
    'Volatilidade Anual (%)': (vol_anual * 100).round(2),
    'Indice Sharpe': sharpe.round(2)
}).sort_values(by='Indice Sharpe', ascending=False)

df_gold.to_csv(f'{BASE_PATH}03_Gold/b3_{segmento}_analise.csv')

print(f"✅ Pipeline {segmento.upper()} Finalizado!")
display(df_gold)

Mounted at /content/drive
🚀 Iniciando Pipeline: COMMODITIES


[*********************100%***********************]  5 of 5 completed


✅ Pipeline COMMODITIES Finalizado!


,Retorno Anualizado (%),Volatilidade Anual (%),Indice Sharpe
PETR4,38.21,31.71,1.21
PRIO3,24.01,40.08,0.60
VALE3,6.14,29.51,0.21
SUZB3,-3.49,28.02,-0.12
CSNA3,-26.92,46.18,-0.58
